# Attention head pairing at the `=` query — a shared motif across runs?

The late-training event in p109 (and p101) shows up at the `=`-query as an **attention-entropy
excursion in two of the four heads** while the other two stay sharp. The head *indices* differ
between models, but the *shape* looks shared. This notebook asks that quantitatively, across three
independently-trained models:

- **p109/s485/ds598**, **p101/s999/ds999**, **p107/s999/ds42**

Heads are permutation-symmetric across random inits, so the only thing that *can* be shared is a
**permutation-invariant** description. Six lenses:

1. **Entropy trajectories** — per-head Shannon entropy of attention at the `=` query (index 2).
2. **Pairing** — per-head excursion magnitude + 4×4 entropy-trajectory correlation.
3. **Functional cross-pair (attention patterns)** — do the co-excursing heads attend alike?
4. **Head → frequency (weight-derived)** — each head's dominant QK frequency, and its stability
   across the event (via the `weight_basis_projection` analyzer — discoverability-first, REQ_107).
5. **Weight-space circuit similarity** — QK frequency content + OV write-subspace, per head pair.
6. **Exploder neurons vs the pairing** — which frequencies does the MLP blow-up live in?

*Conventions follow the SC notebook: all access through the miscope API; forward run in float64
(a measured no-op, kept for consistency).*

In [1]:
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
SPECS = [("p109/s485/ds598", dict(prime=109, seed=485, data_seed=598)),
         ("p101/s999/ds999", dict(prime=101, seed=999, data_seed=999)),
         ("p107/s999/ds42",  dict(prime=107, seed=999, data_seed=42))]
variants = {tag: fam.get_variant(**kw) for tag, kw in SPECS}
EQ_QUERY = 2          # the `=` query position (token order: a, b, =)
POSTGROK = 10000      # correlation window: late training only, excludes the grok ramp
PLATEAU = 20000       # a settled pre-event reference epoch (post-grok, before the late event)
UNIFORM3 = float(np.log(3))   # uniform attention over (a, b, =)
variants

/home/megano/projects/mechinterp/miscope/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'p109/s485/ds598': Variant(family='modulo_addition_1layer', name='p109_seed485_dseed598', state=analyzed),
 'p101/s999/ds999': Variant(family='modulo_addition_1layer', name='p101_seed999_dseed999', state=analyzed),
 'p107/s999/ds42': Variant(family='modulo_addition_1layer', name='p107_seed999_dseed42', state=analyzed)}

## 1. Per-head entropy trajectories

One forward pass per checkpoint per model (~350–400 each — takes a minute or two).

In [2]:
def equals_query_entropy(variant, epochs, query=EQ_QUERY):
    """Per-head Shannon entropy of attention at the `=` query, mean over the (a,b) grid, per epoch."""
    p = variant.params["prime"]
    probe = variant.make_probe([[a, b] for a in range(p) for b in range(p)])
    rows = []
    for e in epochs:
        _, cache = variant.run_with_cache(probe, epoch=int(e), dtype=torch.float64)
        pat = cache["blocks.0.attn.hook_pattern"][:, :, query, :].detach().cpu().numpy()  # (B, H, K)
        rows.append((-(pat * np.log(np.clip(pat, 1e-12, None))).sum(-1)).mean(0))           # (H,)
    return np.array(rows)


traj = {}   # tag -> (epochs, entropy[n_epochs, n_heads])
for tag, v in variants.items():
    eps = np.array(v.get_available_checkpoints())
    traj[tag] = (eps, equals_query_entropy(v, eps))
    print(f"{tag}: {len(eps)} checkpoints, {traj[tag][1].shape[1]} heads")

p109/s485/ds598: 353 checkpoints, 4 heads


p101/s999/ds999: 352 checkpoints, 4 heads


p107/s999/ds42: 403 checkpoints, 4 heads


In [3]:
for tag, (eps, ent) in traj.items():
    fig = go.Figure()
    fig.add_hline(y=UNIFORM3, line=dict(dash="dot", color="gray"),
                  annotation_text="ln 3 (uniform)", annotation_position="top right")
    for h in range(ent.shape[1]):
        fig.add_trace(go.Scatter(x=eps, y=ent[:, h], mode="lines", name=f"head {h}"))
    fig.add_trace(go.Scatter(x=eps, y=ent.mean(1), mode="lines", name="mean",
                             line=dict(color="black", width=3)))
    fig.update_layout(title=f"{tag} — attention entropy at the `=` query",
                      xaxis_title="epoch", yaxis_title="entropy (nats)", height=380,
                      legend=dict(orientation="h", y=-0.25))
    fig.show()

## 2. The pair — excursion magnitude + entropy-trajectory correlation

A genuine pair = two heads whose late-training entropy excursions are both **large** and **tightly
correlated**. The 4×4 correlation is over the post-grok window. Read it together with the per-head
excursion magnitudes: correlation among near-flat heads is just noise, so the pair is the
*large-excursion* block.

In [4]:
def pairing(eps, ent, postgrok=POSTGROK):
    """Identify the excursing head pair permutation-invariantly over the post-grok window."""
    m = eps >= postgrok
    E, pe = ent[m], eps[m]
    base = np.median(E, 0)
    exc = np.abs(E - base).max(0)                       # per-head peak |excursion|
    C = np.corrcoef(E.T)                                # (H, H) trajectory correlation
    pair = tuple(sorted(int(h) for h in np.argsort(exc)[-2:]))
    rest = tuple(h for h in range(ent.shape[1]) if h not in pair)
    within = float(C[pair[0], pair[1]])
    cross = float(np.mean([C[a, b] for a in pair for b in rest]))
    peak_ep = int(pe[np.argmax(np.abs(E - base).mean(1))])
    return dict(exc=exc, C=C, pair=pair, rest=rest, within=within, cross=cross, peak_ep=peak_ep)


pairs = {tag: pairing(eps, ent) for tag, (eps, ent) in traj.items()}

fig = make_subplots(rows=1, cols=len(pairs), subplot_titles=list(pairs))
for j, (tag, P) in enumerate(pairs.items(), start=1):
    n = len(P["exc"])
    fig.add_trace(go.Heatmap(z=P["C"], zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
                             x=[f"h{h}" for h in range(n)], y=[f"h{h}" for h in range(n)],
                             showscale=(j == len(pairs))), row=1, col=j)
fig.update_layout(title="Per-head entropy-trajectory correlation (post-grok window)", height=360)
fig.show()

print(f"{'model':16} {'excursion by head':30} {'pair':8} {'within':>7} {'cross':>7} {'peak_ep':>8}")
for tag, P in pairs.items():
    print(f"{tag:16} {str(np.round(P['exc'], 3)):30} {str(P['pair']):8} "
          f"{P['within']:7.3f} {P['cross']:7.3f} {P['peak_ep']:8d}")

model            excursion by head              pair      within   cross  peak_ep
p109/s485/ds598  [0.067 0.319 0.06  0.402]      (1, 3)     0.976  -0.670    28400
p101/s999/ds999  [0.08  0.464 0.486 0.119]      (1, 2)     0.953  -0.295    29700
p107/s999/ds42   [0.143 0.144 0.405 0.201]      (2, 3)     0.852  -0.483    30400


## 3. Functional cross-pair check — do the co-excursing heads *attend alike*?

The pairing above is *dynamical* (the two heads move together). Probe whether it is *functional* via
cosine similarity between heads' attention-pattern fingerprints (each head's distribution over keys
for every grid input). Evaluate at the **pre-event plateau** (the meaningful test) and at the event
**peak** (where both heads are near-uniform, so high similarity there is a confound).

In [5]:
def head_pattern_cos(variant, epoch, query=EQ_QUERY):
    """Head x head cosine similarity of attention-pattern fingerprints at one epoch."""
    p = variant.params["prime"]
    probe = variant.make_probe([[a, b] for a in range(p) for b in range(p)])
    _, cache = variant.run_with_cache(probe, epoch=int(epoch), dtype=torch.float64)
    pat = cache["blocks.0.attn.hook_pattern"][:, :, query, :].detach().cpu().numpy()  # (B, H, K)
    H = pat.shape[1]
    fp = pat.transpose(1, 0, 2).reshape(H, -1)
    n = np.linalg.norm(fp, axis=1, keepdims=True)
    return (fp @ fp.T) / (n @ n.T)


def plateau_epoch(variant, target=PLATEAU):
    """A settled pre-event plateau checkpoint (fixed target, snapped to nearest available)."""
    eps = np.array(variant.get_available_checkpoints())
    return int(eps[np.argmin(np.abs(eps - target))])


print(f"{'model':16} {'pair':8} | {'plateau pair':>12} {'cross':>7} {'allpairs':>9} | {'peak pair':>10} {'cross':>7}")
for tag, (eps, ent) in traj.items():
    P = pairs[tag]; a, b = P["pair"]; rest = P["rest"]
    Sp = head_pattern_cos(variants[tag], plateau_epoch(variants[tag]))
    Sk = head_pattern_cos(variants[tag], P["peak_ep"])
    H = Sp.shape[0]
    cp = float(np.mean([Sp[a, r] for r in rest] + [Sp[b, r] for r in rest]))
    ck = float(np.mean([Sk[a, r] for r in rest] + [Sk[b, r] for r in rest]))
    allp = float(np.mean(Sp[np.triu_indices(H, 1)]))
    print(f"{tag:16} {str(P['pair']):8} | {Sp[a,b]:12.3f} {cp:7.3f} {allp:9.3f} | {Sk[a,b]:10.3f} {ck:7.3f}")

model            pair     | plateau pair   cross  allpairs |  peak pair   cross


p109/s485/ds598  (1, 3)   |        0.712   0.720     0.693 |      0.815   0.710


p101/s999/ds999  (1, 2)   |        0.706   0.712     0.693 |      0.900   0.764


p107/s999/ds42   (2, 3)   |        0.705   0.701     0.668 |      0.798   0.713


## 4. Head → frequency (weight-derived), and its stability across the event

Each head's dominant QK frequency is the argmax of the marginal QK fractional-power, read from the
`weight_basis_projection` analyzer (the weight-side Fourier projection — not re-derived here). The
question: which two heads share a frequency, and does any head's frequency move through the event?

In [6]:
def head_qk_dominant_freq(variant, epoch):
    """Per-head dominant QK frequency = argmax of the marginal QK fractional-power (weight-derived)."""
    d = variant.artifacts.load_epoch("weight_basis_projection", int(epoch))
    P, f = d["attn_qk_fractional_power"], d["attn_qk_frequencies"]   # (H, F, F), (F,)
    return f[(P.sum(2) + P.sum(1)).argmax(1)]                        # marginal over the pair, per head


for tag, (eps, ent) in traj.items():
    P = pairs[tag]
    wbp_eps = np.array(variants[tag].artifacts.get_epochs("weight_basis_projection"))
    sample = sorted({int(wbp_eps[np.argmin(np.abs(wbp_eps - t))])
                     for t in (PLATEAU, P["peak_ep"], int(wbp_eps[-1]))})
    freqs = {e: head_qk_dominant_freq(variants[tag], e).tolist() for e in sample}
    f0 = freqs[sample[0]]
    pair_f = sorted({f0[h] for h in P["pair"]})
    flat_f = sorted({f0[h] for h in P["rest"]})
    stable = all(freqs[e] == f0 for e in sample)
    print(f"{tag}: head->freq {f0}")
    print(f"   excursing {P['pair']} = freq {pair_f} (distinct) | "
          f"flat {P['rest']} = freq {flat_f} ({'SAME' if len(flat_f) == 1 else 'distinct'})")
    print(f"   frequency assignment stable across event: {stable}  (epochs {sample})")

p109/s485/ds598: head->freq [4, 27, 4, 14]
   excursing (1, 3) = freq [14, 27] (distinct) | flat (0, 2) = freq [4] (SAME)
   frequency assignment stable across event: True  (epochs [20000, 28400, 34999])


p101/s999/ds999: head->freq [36, 50, 26, 36]
   excursing (1, 2) = freq [26, 50] (distinct) | flat (0, 3) = freq [36] (SAME)
   frequency assignment stable across event: True  (epochs [20000, 29700, 34999])


p107/s999/ds42: head->freq [52, 52, 18, 46]
   excursing (2, 3) = freq [18, 46] (distinct) | flat (0, 1) = freq [52] (SAME)
   frequency assignment stable across event: True  (epochs [20000, 30400, 39999])


## 5. Weight-space circuit similarity — QK frequency content + OV write-subspace

Two basis-invariant, head×head similarities at the **pre-event plateau** (epoch 20000):

- **QK frequency content** — cosine of the `attn_qk_fractional_power` maps (do two heads attend with
  the same frequencies?).
- **OV write-subspace** — mean squared cosine of principal angles between heads' `W_O` row-spaces,
  in [0,1] (random baseline ≈ d_head/d_model). Does the raw write-side agree?

If the flat pair is frequency-redundant, it should be ~1 on QK; the OV column tells us whether the
redundancy is read-side only.

In [7]:
def qk_freq_similarity(variant, epoch):
    """Head x head cosine of QK fractional-power maps (basis-invariant frequency content)."""
    P = variant.artifacts.load_epoch("weight_basis_projection", int(epoch))["attn_qk_fractional_power"]
    H = P.shape[0]; fp = P.reshape(H, -1)
    n = np.linalg.norm(fp, axis=1, keepdims=True)
    return fp @ fp.T / (n @ n.T)


def ov_subspace_overlap(variant, epoch):
    """Head x head OV write-subspace overlap from raw W_O (mean sq cos of principal angles, in [0,1])."""
    Wo = variant.artifacts.load_epoch("parameter_snapshot", int(epoch))["W_O"]   # (H, d_head, d_model)
    H, dh, dm = Wo.shape
    Q = [np.linalg.qr(Wo[h].T)[0] for h in range(H)]                              # (d_model, d_head)
    S = np.array([[np.linalg.norm(Q[i].T @ Q[j]) ** 2 / dh for j in range(H)] for i in range(H)])
    return S, dh / dm


fig = make_subplots(rows=len(pairs), cols=2, subplot_titles=[
    f"{tag} — {kind}" for tag in pairs for kind in ("QK frequency content", "OV write-subspace")])
print(f"{'model':16} {'exc pair':9} | {'QK flat':>8} {'QK exc':>7} {'QK cross':>8} | "
      f"{'OV flat':>8} {'OV exc':>7} {'OV cross':>8} {'rand':>6}")
for i, (tag, P) in enumerate(pairs.items(), start=1):
    plat = plateau_epoch(variants[tag])
    Sqk = qk_freq_similarity(variants[tag], plat)
    Sov, rand = ov_subspace_overlap(variants[tag], plat)
    n = Sqk.shape[0]; ticks = [f"h{h}" for h in range(n)]
    fig.add_trace(go.Heatmap(z=Sqk, zmin=0, zmax=1, colorscale="Viridis", x=ticks, y=ticks,
                             showscale=(i == 1)), row=i, col=1)
    fig.add_trace(go.Heatmap(z=Sov, zmin=0, zmax=1, colorscale="Viridis", x=ticks, y=ticks,
                             showscale=False), row=i, col=2)
    a, b = P["pair"]; rest = P["rest"]   # `pair` = excursing (distinct freq); `rest` = flat (same freq)
    def trio(S):
        flat = S[rest[0], rest[1]]; exc = S[a, b]
        cross = float(np.mean([S[x, y] for x in P["pair"] for y in rest]))
        return flat, exc, cross
    qf, qe, qc = trio(Sqk); of, oe, oc = trio(Sov)
    print(f"{tag:16} {str(P['pair']):9} | {qf:8.3f} {qe:7.3f} {qc:8.3f} | "
          f"{of:8.3f} {oe:7.3f} {oc:8.3f} {rand:6.2f}")
fig.update_layout(title="Head x head circuit similarity at the pre-event plateau (flat = same-frequency pair)",
                  height=360 * len(pairs))
fig.show()

model            exc pair  |  QK flat  QK exc QK cross |  OV flat  OV exc OV cross   rand
p109/s485/ds598  (1, 3)    |    0.999   0.003    0.023 |    0.360   0.298    0.328   0.25


p101/s999/ds999  (1, 2)    |    0.986   0.007    0.071 |    0.351   0.295    0.322   0.25
p107/s999/ds42   (2, 3)    |    0.999   0.003    0.025 |    0.341   0.317    0.328   0.25


## 6. Exploder neurons vs the pairing — which frequencies blow up?

§1–§5 are read off attention. This section asks the MLP-side question: the ~handful of neurons whose
`W_in` rows explode across the event (notebook-1's "exploders") — do they live in the **excursing
(solo)** frequencies, the **doubled (flat-pair)** frequency, or **off** the committed set entirely?

Exploders = top-k neurons by peak `W_in` event-excursion in units of plateau drift (notebook-1 §1
definition). Each neuron's dominant frequency is read from `weight_basis_projection.mlp_in`. Compared
against the **base rate** (the same fractions over *all* 512 neurons), so a frequency that simply
holds most neurons doesn't masquerade as enrichment.

In [8]:
def exploder_neurons(variant, peak_ep, k=11, postgrok=POSTGROK):
    """Top-k MLP neurons by peak W_in event-excursion in units of plateau drift (notebook-1 §1)."""
    eps = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
    Win = np.stack([variant.artifacts.load_epoch("parameter_snapshot", int(e))["W_in"].T.astype(np.float64)
                    for e in eps])                                  # (E, N, d_model)
    plat = (eps >= postgrok) & (eps <= peak_ep - 2000)
    evt = (eps >= peak_ep - 1500) & (eps <= peak_ep + 1500)
    ref = Win[plat].mean(0)
    drift = np.linalg.norm(np.diff(Win[plat], axis=0), axis=2).mean(0) + 1e-9
    excursion = np.linalg.norm(Win[evt] - ref[None], axis=2).max(0)
    return np.argsort(excursion / drift)[::-1][:k]


def neuron_dominant_freq(variant, epoch):
    """Per-neuron dominant input-side frequency (weight_basis_projection.mlp_in)."""
    return variant.artifacts.load_epoch("weight_basis_projection", int(epoch))["mlp_in_dominant_frequency"].ravel()


cats = ["solo (excursing)", "doubled (flat)", "off-committed"]
fig = make_subplots(rows=1, cols=len(pairs), subplot_titles=list(pairs))
print(f"{'model':16} | {'solo: expl':>10} {'base':>6} | {'doubled: expl':>13} {'base':>6} | "
      f"{'off-comm: expl':>14} {'base':>6}")
for j, (tag, P) in enumerate(pairs.items(), start=1):
    plat = plateau_epoch(variants[tag])
    fhead = head_qk_dominant_freq(variants[tag], plat)
    solo = {int(fhead[h]) for h in P["pair"]}        # excursing pair frequencies (distinct)
    doubled = {int(fhead[h]) for h in P["rest"]}     # flat pair frequency (one value)
    committed = {int(x) for x in fhead}
    ex = exploder_neurons(variants[tag], P["peak_ep"])
    nf = neuron_dominant_freq(variants[tag], plat)
    ef = nf[ex]
    frac = lambda sel, arr: float(np.mean(np.isin(arr, list(sel))))
    off = lambda arr: float(np.mean(~np.isin(arr, list(committed))))
    ex_v = [frac(solo, ef), frac(doubled, ef), off(ef)]
    base_v = [frac(solo, nf), frac(doubled, nf), off(nf)]
    fig.add_trace(go.Bar(x=cats, y=ex_v, name="exploders", marker_color="#d62728",
                         showlegend=(j == 1)), row=1, col=j)
    fig.add_trace(go.Bar(x=cats, y=base_v, name="all neurons (base)", marker_color="gray",
                         showlegend=(j == 1)), row=1, col=j)
    print(f"{tag:16} | {ex_v[0]:10.0%} {base_v[0]:6.0%} | {ex_v[1]:13.0%} {base_v[1]:6.0%} | "
          f"{ex_v[2]:14.0%} {base_v[2]:6.0%}    exploders={ex.tolist()} freqs={ef.tolist()}")
fig.update_layout(title="Exploder frequency membership vs base rate (by head-pairing role)",
                  barmode="group", height=380, yaxis_title="fraction of neurons")
fig.show()

model            | solo: expl   base | doubled: expl   base | off-comm: expl   base


p109/s485/ds598  |        36%    55% |           64%    45% |             0%     0%    exploders=[327, 400, 326, 420, 98, 303, 414, 421, 80, 176, 63] freqs=[4, 4, 4, 4, 14, 4, 4, 27, 27, 4, 27]


p101/s999/ds999  |        45%    56% |           36%    41% |            18%     3%    exploders=[244, 247, 40, 166, 335, 74, 379, 461, 86, 391, 374] freqs=[50, 26, 36, 36, 26, 36, 36, 29, 29, 50, 50]


p107/s999/ds42   |        36%    59% |           64%    38% |             0%     4%    exploders=[364, 375, 427, 474, 381, 248, 261, 208, 437, 373, 119] freqs=[52, 52, 18, 46, 52, 46, 46, 52, 52, 52, 52]


## Reading

Putting the lenses together, the late-training event has a clean structural signature that
**reproduces across three independently-trained models**.

**The heads split 2+2 by frequency redundancy.** Each model commits **3 frequencies across its 4
heads, with one doubled** (p109 {4,4,14,27}; p101 {26,36,36,50}; p107 {18,46,52,52}). In frequency
space the two heads sharing a frequency are **near-identical QK circuits** (QK fractional-power
cosine **≈0.99–1.0**); the distinct-frequency heads are **QK-orthogonal** (~0.003–0.07). The
redundancy is **read-side**: OV write-subspace overlap barely separates the pairs (flat ≈ cross,
both just above the random baseline). So two heads attend *with the same frequency*, but do not
obviously *write the same place*.

**The redundant pair is the stable one; the unique-frequency heads carry the attention event.** The
`=`-query entropy excursion (§1–§2) lands on the **distinct-frequency pair** (within-pair entropy
correlation 0.85–0.98) while the **doubled-frequency pair stays sharp** — an anti-phase 2+2 see-saw
(negative cross-correlation). The excursion is a *rise* toward uniform (de-sharpening), the opposite
of entropy collapse, and it round-trips.

**Frequency identity is invariant through the event (§4).** No head's dominant QK frequency moves
across the instability window. The event softens *attention sharpness* in the solo-frequency heads,
not the *frequency assignment* — the model's frequency bookkeeping is untouched.

**But the MLP blow-up lives in a *different* frequency than the attention excursion (§6).** The
exploder neurons are **depleted in the solo/excursing frequencies** in all three runs (36/45/36% vs
55/56/59% base) and, in p109 and p107, **concentrated in the doubled/flat frequency** (64% vs 45/38%
base — p109's top exploders n327, n400 are both the doubled freq). p101 is mixed, including
**off-committed** exploders (e.g. freq 29). So the event is two-faced in *complementary* frequency
sets: the *solo-frequency heads* de-sharpen on the read side, while the *doubled-frequency neurons*
blow up on the write side — the redundant frequency's heads stay sharp even as its neurons explode.
The blow-up favors the most heavily-resourced frequency (two heads **and** a plurality of neurons),
not the destabilizing solo ones.

**Why §3 looked null.** Attention-pattern cosine over the three keys {a, b, =} cannot see frequency
content — that lives in the QK Fourier structure (§5). The "functional pairing" is real, but it is a
**QK-frequency** pairing, recovered only in weight/Fourier space, not in the attention distribution.

**Caveats.** N = 3 runs, 4 heads, 11 exploders. The strong, reproducible parts: (a) the
doubled-frequency QK redundancy (cos ≈ 1 vs ≈ 0), (b) the excursing pair is the distinct-frequency
pair, (c) frequency-assignment invariance through the event, (d) exploders depleted in the solo
frequencies. The doubled-frequency *enrichment* of exploders is suggestive, not significant at N=11;
the robust §6 signal is the solo-frequency *depletion*. The *mechanism* — why solo-frequency heads
de-sharpen while the redundant anchor and its (exploding) neurons hold the function — is
characterized, not explained. Each prime+seed+data_seed is a unique model run; the claim is the
motif's reproduction across three, not that the head indices generalize.